# Pretrained Qwen3.5-9B-Base: J-lens vs handwritten logit lens

**Question:** Does the supplied pretrained Jacobian lens expose annotated intermediates
better than a direct logit lens on the same residuals? **Answer:** Not yet measured in this
notebook; run it to populate the tables and plots. No results are fabricated here.

Derived from `model_agnostic_lens_dataset.ipynb`; the original notebook and its saved
outputs are unchanged. This notebook **only downloads/loads a pretrained lens: no fitting,
calibration corpus, or Drive setup**. Defaults: `Qwen/Qwen3.5-9B-Base` and the supplied
Neuronpedia 9B lens. Change the model and matching lens together. A matching shape is not
proof of compatible model revision, fitting dtype, BOS policy, or training corpus.

The base model receives raw prompts, not a chat template. fp32 9B weights alone need
roughly 36 GB, plus substantial activation/logit and lens memory; use a large GPU/RAM
runtime. bf16 is configurable, but changes numerical comparisons. Install the repository
with its dependencies first (`pip install -e .`); Transformers >=5.5 is required.

## Scoring protocol: complete decoded tokens, not generated answers

* Preserve **all prompt whitespace**. Read out the final prompt token; poetry uses the
  last token containing a newline, as in the dataset protocol.
* Build accepted token IDs from **exact tokenizer-decoded spellings**, never raw vocabulary
  marker strings. Accept as-is/lower/capitalized variants and zero/one leading ASCII space.
  Order-ops uses the same number/operation synonyms for **both target ranks and correctness**.
* Never score a whitespace token or a first-token prefix as a complete target. Words with
  no complete accepted token are **unsupported**: null ranks and null correctness, not errors
  or misses. Show target and word coverage before interpreting supported-only plots.
* Rank is 1-based against the **full vocabulary**, minimized across accepted IDs. Ties use
  strictly greater logits; tied rank 1 does not necessarily equal the chosen argmax token.
* pass@k averages supported-word hit fractions per item, then across items with supported
  words. Unsupported words/items are excluded; therefore denominators differ from legacy
  first-token-fallback results. Controls are words from another item of the same dataset.
* `model_correct` means **next-token lexical match on supported targets**, not complete-answer
  accuracy, contextual tokenization likelihood, arithmetic ability, or generated-answer accuracy.
  Accepting a single-token synonym does not establish that a multi-token answer was generated.
* BOS is explicit below and shared by evaluation, HF check, and interactive slices. No
  silent prompt truncation; overlength inputs fail. Behavioral metrics exclude special positions.

The Qwen glossary is machine-generated, display-only metadata. It changes neither accepted
spellings nor scoring. Its token-ID provenance is unverified; disable it for replacement models.

## Setup

In [ ]:
import os
import subprocess
import sys
from pathlib import Path

REPO_URL = "https://github.com/ai360-project-school-j-lens/jacobian-lens-gpt2.git"
local_repo = next(
    (p for p in [Path.cwd(), *Path.cwd().parents]
     if (p / "jlens" / "evaluation.py").is_file()), None,
)
if local_repo is None:
    local_repo = Path.cwd() / "jacobian-lens-gpt2"
    if not local_repo.exists():
        subprocess.run(["git", "clone", REPO_URL, str(local_repo)], check=True)
REPO_DIR = str(local_repo.resolve())
sys.path.insert(0, REPO_DIR)
if not (Path(REPO_DIR) / "jlens" / "strict_scoring.py").is_file():
    raise RuntimeError("Use the repository revision containing this notebook and strict_scoring.py")

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from IPython.display import display
from transformers import AutoModelForCausalLM, AutoTokenizer

import jlens
from jlens import ActivationRecorder, HFLensModel, LensModel
from jlens.evaluation import (
    DATASETS,
    ROLE_NAMES,
    evaluate_paired,
    first_layer,
    identity_lens,
    layer_hit_rate,
    layer_mean,
    layer_median_rank,
    load_eval,
    pass_at_k,
    plot_layer_curves,
    shown_layers,
)

pd.set_option("display.max_columns", 80)
pd.set_option("display.width", 240)
from jlens.strict_scoring import DecodedSpellings, ExplicitPromptModel

In [ ]:
MODEL_ID = "Qwen/Qwen3.5-9B-Base"
MODEL_REVISION = "main"  # Pin a commit for reproducibility (model AND tokenizer).
LENS_URL = "https://huggingface.co/neuronpedia/jacobian-lens/resolve/main/qwen3.5-9b-pt/jlens/Salesforce-wikitext/Qwen3.5-9B-Base_jacobian_lens.pt"
LOCAL_LENS_PATH = None  # Optional existing .pt file; bypasses the URL.
CACHE_DIR = Path.home() / ".cache" / "jlens-pretrained"
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
DTYPE = torch.float32  # Option: torch.bfloat16 on a compatible accelerator.
BOS_POLICY = "none"  # Qwen base raw text. Options: "none", "prepend", "tokenizer".
ENABLE_QWEN_GLOSS = MODEL_ID == "Qwen/Qwen3.5-9B-Base"
USE_RESULT_CACHE = True  # Only read locally generated, trusted pickle files.

hf_model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID, revision=MODEL_REVISION, dtype=DTYPE,
).to(DEVICE)
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, revision=MODEL_REVISION)
adapter = jlens.from_hf(hf_model, tokenizer, compile=False, force_bos=False)
assert isinstance(adapter, HFLensModel)
model: LensModel = ExplicitPromptModel(adapter, bos_policy=BOS_POLICY)
assert model.n_layers >= 2
spellings = DecodedSpellings(
    tokenizer, vocab_size=hf_model.get_output_embeddings().weight.shape[0],
)
STRICT_SCORING = dict(spelling_lookup=spellings, preserve_prompt_whitespace=True)
LENS_NAMES = ("logit lens", "J-lens")
LAYER_STRIDE = max(1, model.n_layers // 12)
print(adapter)
print("BOS policy:", BOS_POLICY, "tokenizer:", type(tokenizer).__name__)
print("Probe IDs:", model.encode("2 + 2 = ")[0].tolist())

In [ ]:
evals = {dataset: load_eval(REPO_DIR, dataset) for dataset in DATASETS}

K = 10
KS = sorted({1, 5, 10, 100, K})
LAST = model.n_layers - 1
{dataset: len(samples) for dataset, samples in evals.items()}

## Handwritten logit lens on `LensModel`

`ActivationRecorder` hooks the output of every block **before the final normalization**.
`model.unembed` applies the adapter's final norm, output head, and any logit softcap.
This avoids assuming a particular `hidden_states` convention or normalizing the final output twice.

The paired evaluator calls `handwritten_logit_lens` with activations it has already recorded;
it then reads the J-lens from that same dictionary. The standalone `logit_lens` wrapper is useful
for custom prompts, but is not called separately during dataset evaluation.

In [ ]:
@torch.no_grad()
def handwritten_logit_lens(model: LensModel, activations: dict[int, torch.Tensor]) -> torch.Tensor:
    """[n_layers, seq_len, vocab]: direct readout of every raw block output."""
    logits = []
    for layer in range(model.n_layers):
        residual = activations[layer][0].float()
        logits.append(model.unembed(residual).float())
    return torch.stack(logits)


@torch.no_grad()
def logit_lens(model: LensModel, prompt: str) -> torch.Tensor:
    """One-prompt handwritten logit lens, using only the original jlens interface."""
    with ActivationRecorder(model.layers, at=range(model.n_layers)) as recorder:
        model.forward(model.encode(prompt))
    return handwritten_logit_lens(model, recorder.activations)

### Final-layer check against the HuggingFace model

On one prompt, verify that the handwritten final row is the model's own output. This separate
sanity check runs two forwards; the dataset evaluator below still uses one forward per prompt.

In [ ]:
check_prompt = evals["order-ops"][0]["prompt"]
with torch.no_grad():
    final_handwritten = logit_lens(model, check_prompt)[-1]
    final_hf = hf_model(input_ids=model.encode(check_prompt), use_cache=False).logits[0].float()
torch.testing.assert_close(final_handwritten, final_hf, rtol=1e-4, atol=1e-5)
print("Handwritten final layer matches the HuggingFace model.")
del final_handwritten, final_hf

## Cached pretrained lens — no fitting

Use the supplied HTTPS URL or `LOCAL_LENS_PATH`. URL-keyed files are downloaded once,
atomically, then reused without a network request. Delete the cached file explicitly to
refresh a mutable URL; prefer a commit-pinned URL for reproducibility. Loading uses
`JacobianLens.load` (`torch.load(weights_only=True)`), not a fitting checkpoint loader.

The checks reject missing inner layers, non-finite/misshaped matrices, and width mismatch.
Any supplied final-layer matrix is ignored: the comparison must use the actual model output.
External fitting provenance remains **UNVERIFIED**. The run records a lens content hash,
model/tokenizer revision information, dtype, BOS policy, and scoring definition.

In [ ]:
import hashlib
import json
import urllib.parse
import urllib.request

CACHE_DIR.mkdir(parents=True, exist_ok=True)
if LOCAL_LENS_PATH is not None:
    lens_path = Path(LOCAL_LENS_PATH).expanduser()
    if not lens_path.is_file():
        raise FileNotFoundError(lens_path)
else:
    if urllib.parse.urlparse(LENS_URL).scheme != "https":
        raise ValueError("Use an HTTPS lens URL")
    url_key = hashlib.sha256(LENS_URL.encode()).hexdigest()[:20]
    lens_path = CACHE_DIR / f"lens-{url_key}.pt"
    if not lens_path.is_file():
        temporary = lens_path.with_suffix(".download")
        try:
            urllib.request.urlretrieve(LENS_URL, temporary)
            temporary.replace(lens_path)
        finally:
            temporary.unlink(missing_ok=True)
    else:
        print("Using cached lens:", lens_path)

fitted_lens = None  # Clear stale state before loading/checking.
candidate = jlens.JacobianLens.load(str(lens_path))
if candidate.d_model != model.d_model:
    raise ValueError("Lens/model residual widths differ")
required = set(range(model.n_layers - 1))
if required - set(candidate.source_layers):
    raise ValueError(f"Missing inner layers: {sorted(required - set(candidate.source_layers))}")
if set(candidate.source_layers) - set(range(model.n_layers)):
    raise ValueError("Lens contains out-of-range layer indices")
for layer_id, matrix in candidate.jacobians.items():
    if matrix.shape != (model.d_model, model.d_model) or not torch.isfinite(matrix).all():
        raise ValueError(f"Invalid Jacobian at layer {layer_id}")
if model.n_layers - 1 in candidate.jacobians:
    print("Ignoring supplied final-layer Jacobian; using the model output.")
fitted_lens = jlens.JacobianLens(
    {layer_id: candidate.jacobians[layer_id] for layer_id in sorted(required)},
    n_prompts=candidate.n_prompts, d_model=candidate.d_model,
)
del candidate
print("UNVERIFIED external fitting provenance:", fitted_lens)

In [ ]:
def file_sha256(path):
    digest = hashlib.sha256()
    with open(path, "rb") as handle:
        for chunk in iter(lambda: handle.read(1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()

import transformers

run_metadata = {
    "model": MODEL_ID, "requested_revision": MODEL_REVISION,
    "resolved_model_commit": getattr(hf_model.config, "_commit_hash", None),
    "tokenizer_class": type(tokenizer).__name__,
    "tokenizer_revision": tokenizer.init_kwargs.get("_commit_hash"),
    "tokenizer_backend_sha256": hashlib.sha256(
        tokenizer.backend_tokenizer.to_str().encode()
    ).hexdigest() if hasattr(tokenizer, "backend_tokenizer") else None,
    "dtype": str(DTYPE), "device": str(DEVICE), "bos_policy": BOS_POLICY,
    "lens_url": LENS_URL if LOCAL_LENS_PATH is None else None,
    "lens_sha256": file_sha256(lens_path), "lens_n_prompts": fitted_lens.n_prompts,
    "fitting_provenance": "UNVERIFIED", "scoring": "decoded-whole-token-v1",
    "prompt_whitespace": "preserved", "torch": torch.__version__,
    "transformers": transformers.__version__,
    "matmul_precision": torch.get_float32_matmul_precision(),
    "cuda_matmul_tf32": torch.backends.cuda.matmul.allow_tf32,
    "cudnn_tf32": torch.backends.cudnn.allow_tf32,
}
display(run_metadata)

### Identity sanity check

With `J_l = I`, both readouts must produce the same ranks, best layers, top-1 tokens,
agreement, copy rate, and KL. This small check uses the same paired evaluator as the full run.
TF32 and autocast are disabled only within this **exact** assertion, restored even if it fails.
The configured model dtype is unchanged: both paths use the same adapter unembedding;
identity transport operates on fp32 residuals. No silent weight casting or relaxed assertion.

In [ ]:
sanity_evals = {"order-ops": evals["order-ops"][:2]}
from jlens.notebook_setup import strict_identity_precision

with strict_identity_precision(model.input_device.type):
    sanity_words, sanity_items = evaluate_paired(model, identity_lens(model), sanity_evals, desc='identity check', logit_readout=handwritten_logit_lens, **STRICT_SCORING)
    for frame in (sanity_words, sanity_items):
        left = frame[frame.lens == 'logit lens'].drop(columns='lens').reset_index(drop=True)
        right = frame[frame.lens == 'J-lens'].drop(columns='lens').reset_index(drop=True)
        pd.testing.assert_frame_equal(left, right, check_exact=True)
del sanity_words, sanity_items
print("Identity lens: all metrics match.")

## 1. Paired evaluation and supported-token coverage

One shared forward per prompt, with a locally cached table bundle (no vocabulary logits).
The cache key covers the dataset, run metadata, notebook, and relevant implementation files.
Only open **your own trusted local pickle cache**. Delete it or set `USE_RESULT_CACHE=False`
to recompute. Identity/final-output sanity checks still run independently.

`all_words` retains unsupported rows with null ranks; `words` contains only supported rows
for the existing plotting functions. Item predictions retain raw decoded spellings and
original prompt whitespace; glossary translations are never re-encoded for scoring.

In [ ]:
import pickle

cache_description = {
    "run": run_metadata, "evals": evals,
    "implementation": {
        name: file_sha256(Path(REPO_DIR) / name) for name in (
            "jlens/evaluation.py", "jlens/strict_scoring.py", "jlens/hf.py",
            "jlens/hooks.py", "jlens/lens.py",
        )
    },
}
notebook_path = Path(REPO_DIR) / "notebooks/jacobian_lens/pretrained_qwen_lens_dataset.ipynb"
notebook_code = [
    cell["source"] for cell in json.loads(notebook_path.read_text())["cells"]
    if cell["cell_type"] == "code"
]
cache_description["notebook_code_sha256"] = hashlib.sha256(
    json.dumps(notebook_code).encode()
).hexdigest()  # Saving outputs does not invalidate the inference cache.
cache_key = hashlib.sha256(
    json.dumps(cache_description, sort_keys=True).encode()
).hexdigest()[:24]
result_path = CACHE_DIR / f"paired-{cache_key}.pkl"
if USE_RESULT_CACHE and result_path.is_file():
    with result_path.open("rb") as handle:
        saved = pickle.load(handle)  # Trusted local artifacts only.
    if saved["description"] != cache_description:
        raise ValueError("Result cache metadata mismatch")
    all_words, items = saved["words"], saved["items"]
    print("Loaded paired results:", result_path)
else:
    all_words, items = evaluate_paired(
        model, fitted_lens, evals, logit_readout=handwritten_logit_lens,
        **STRICT_SCORING,
    )
    if USE_RESULT_CACHE:
        temporary = result_path.with_suffix(".tmp")
        try:
            with temporary.open("wb") as handle:
                pickle.dump({"description": cache_description, "words": all_words, "items": items}, handle)
            temporary.replace(result_path)
        finally:
            temporary.unlink(missing_ok=True)

word_coverage = all_words[all_words.lens == "logit lens"].assign(
    supported=lambda frame: frame.ranks.map(lambda rank: rank is not None),
).groupby(["dataset", "kind"]).supported.agg(["sum", "size", "mean"])
word_coverage.columns = ["supported", "total", "coverage"]
display(word_coverage)
words = all_words[all_words.ranks.map(lambda rank: rank is not None)].copy()
if words.empty:
    raise ValueError("No supported words: inspect the tokenizer and coverage table")

words["inner_best"] = [int(r[:-1].min()) for r in words.ranks]
words["inner_best_layer"] = [int(r[:-1].argmin()) for r in words.ranks]

# The final row and model answers must agree for every paired item/word.
word_keys = ["dataset", "item", "kind", "word", "role"]
final_ranks = words.assign(final_rank=[int(r[-1]) for r in words.ranks]).pivot(
    index=word_keys, columns="lens", values="final_rank"
)
assert (final_ranks["logit lens"] == final_ranks["J-lens"]).all()
for column in ("model_top1", "model_correct", "readout_token"):
    paired = items.pivot(index=["dataset", "item"], columns="lens", values=column)
    pd.testing.assert_series_equal(paired["logit lens"], paired["J-lens"], check_names=False)
for name in LENS_NAMES:
    frame = items[items.lens == name]
    np.testing.assert_allclose(np.stack(frame.agreement)[:, -1], 1)
    np.testing.assert_allclose(np.stack(frame.kl_to_final)[:, -1], 0, atol=1e-6)
print("Final model readout matches for both lenses.")

In [ ]:
words.head()

In [ ]:
items.head()

In [ ]:
inter = words[words.kind == "intermediate"]
control = words[words.kind == "control"]
target = words[words.kind == "target"]

## 2. Next-token target matching (supported targets only)

Count each item once, not once per lens. Missing annotations and unsupported targets remain
null, and have separate counts below. Coverage is scored/annotated, not scored/all items.
This is a lexical next-token probe, **not generated-answer task accuracy**.

In [ ]:
model_items = items.drop_duplicates(["dataset", "item"])
with_target = model_items.dropna(subset=["model_correct"]).astype({"model_correct": bool})
accuracy = with_target.groupby("dataset").model_correct.mean()
coverage = model_items.assign(
    annotated=model_items.target.notna(),
    scored=model_items.model_correct.notna(),
).groupby("dataset").agg(total=("item", "size"), annotated=("annotated", "sum"), scored=("scored", "sum"))
coverage["unsupported"] = coverage.annotated - coverage.scored
coverage["unannotated"] = coverage.total - coverage.annotated
coverage["coverage"] = coverage.scored / coverage.annotated.replace(0, np.nan)
coverage["next_token_match_supported"] = accuracy
coverage["correct"] = with_target.groupby("dataset").model_correct.sum().reindex(coverage.index, fill_value=0)
coverage["incorrect"] = coverage.scored - coverage.correct
display(coverage)

In [ ]:
preview = model_items.groupby("dataset").head(6).copy()
# repr makes spaces/newlines visible; raw strings are still stored in items.
preview["prediction_repr"] = preview.model_top1.map(repr)
preview["prompt_repr"] = preview.prompt.map(repr)
display(preview[["dataset", "item", "prompt_repr", "target", "prediction_repr", "model_correct"]])

## 3. pass@k: intermediates vs control vs target

In [ ]:
scores = pass_at_k(words, ks=KS)
scores.pivot_table(index="dataset", columns=["kind", "k", "lens"], values="score").round(3)

The min over layers includes the last row — the model's own output. Split it: what do the inner layers (all but the last)
show that the output does not?

In [ ]:
inner = pass_at_k(words, ks=KS, layers=slice(None, -1)).assign(readout=f"layers 0..{LAST - 1}")
output = pass_at_k(words, ks=KS, layers=[LAST]).assign(readout="model output")
split = pd.concat([inner, output], ignore_index=True)
split[split.k == K].pivot_table(index="dataset", columns=["kind", "readout", "lens"], values="score").round(3)

In [ ]:
at_k = scores[scores.k == K].pivot(index="dataset", columns=["kind", "lens"], values="score")
at_k.plot.bar(figsize=(13, 4), rot=0, title=f"Both lenses: pass@{K} (min over all layers)")
plt.ylabel(f"pass@{K}")
plt.show()

inner_at_k = inner[inner.k == K].pivot(index="dataset", columns=["kind", "lens"], values="score")
output_at_k = output[output.k == K].pivot(index="dataset", columns=["kind", "lens"], values="score")
inner_delta = pd.DataFrame({
    kind: inner_at_k[(kind, "J-lens")] - inner_at_k[(kind, "logit lens")]
    for kind in ("intermediate", "control", "target")
})
display(inner_delta.rename_axis(columns="J-lens − logit lens, inner pass@k").round(3))

## 4. At what depth do the words surface?

In [ ]:
def word_curves(stat):
    return {
        f"{name}: {kind}": stat(words[(words.lens == name) & (words.kind == kind)])
        for name in LENS_NAMES
        for kind in ("intermediate", "control", "target")
    }


plot_layer_curves(
    word_curves(lambda frame: layer_hit_rate(frame, K)),
    title=f"Both lenses: share of words with rank ≤ {K} at the readout position",
    ylabel=f"rank ≤ {K}",
)
plot_layer_curves(
    word_curves(layer_median_rank),
    title="Both lenses: median rank at the readout position",
    ylabel="median rank",
    logy=True,
)

Where the best rank of each intermediate is reached (only words that are hits, rank ≤ K):

In [ ]:
(
    inter[inter.best_rank <= K]
    .groupby(["dataset", "lens"]).best_layer.value_counts().unstack(fill_value=0)
    .reindex(columns=range(model.n_layers), fill_value=0)
)

## 5. How both lenses behave in general

Over all prompt positions (not only the readout one):

* **agreement** — the layer's top-1 equals the model's top-1: how deep the final prediction is formed;
* **copy rate** — the layer's top-1 is the input token at that position: the residual stream still "is" the token embedding
  (models with tied embeddings share the input and output token weights, so an untouched embedding decodes to itself);
* **KL(model ‖ layer)** — how far the layer's distribution is from the output one.

In [ ]:
behaviour = {
    name: {column: layer_mean(items[items.lens == name], column)
           for column in ("agreement", "copy_rate", "kl_to_final")}
    for name in LENS_NAMES
}
plot_layer_curves(
    {f"{name}: top-1 == model top-1": behaviour[name]["agreement"] for name in LENS_NAMES}
    | {f"{name}: top-1 == input token": behaviour[name]["copy_rate"] for name in LENS_NAMES},
    title="Both lenses over all prompt positions",
    ylabel="share of positions",
)
plot_layer_curves(
    {f"{name}: KL(model || layer)": behaviour[name]["kl_to_final"] for name in LENS_NAMES},
    title="Both lenses: KL to the model output",
    ylabel="nats",
)

In [ ]:
behaviour_rows = []
for name in LENS_NAMES:
    agreement, copy_rate, kl = (behaviour[name][c] for c in ("agreement", "copy_rate", "kl_to_final"))
    for dataset in agreement.index:
        behaviour_rows.append({
            "dataset": dataset,
            "lens": name,
            "agreement ≥ 50% from": first_layer(agreement.loc[dataset], 0.5),
            "KL ≤ 1 nat from": first_layer(kl.loc[dataset], 1.0, above=False),
            "copy rate at L0": copy_rate.loc[dataset, 0],
            f"copy rate at L{LAST - 1}": copy_rate.loc[dataset, LAST - 1],
        })
pd.DataFrame(behaviour_rows).set_index(["dataset", "lens"]).unstack("lens").round(3)

## 6. What drives a hit

### 6a. The word is already in the prompt

Early layers of the logit lens can decode the input token itself, so a word that is literally in the prompt
(e.g. a number in order-ops) can be a "hit" without any computation.

In [ ]:
hits = inter.assign(hit=inter.best_rank <= K, inner_hit=inter.inner_best <= K)
hits.groupby(["dataset", "in_prompt", "lens"]).hit.agg(["mean", "size"]).unstack(["in_prompt", "lens"]).round(3)

### 6b. Unsupported words (never first-token fallbacks)

These entries are excluded from rank plots, not counted as false answers. The coverage table
above gives full denominators. Supported order-ops synonyms may still permit a lexical probe
when the literal numeric spelling itself has no complete token.

In [ ]:
unsupported_words = all_words[
    (all_words.lens == "logit lens") & all_words.ranks.map(lambda rank: rank is None)
]
display(unsupported_words[["dataset", "item", "kind", "word", "role"]].head(30))

### 6c. Role of the intermediate

multilingual: `[language, relation, source word, answer in English]`; order-ops: `[intermediate number, operation]`.

In [ ]:
roles = hits[hits.dataset.isin(list(ROLE_NAMES))].copy()
roles["role_name"] = [ROLE_NAMES[d][r] for d, r in zip(roles.dataset, roles.role)]
roles.groupby(["dataset", "role_name", "lens"]).agg(
    hit_rate=("hit", "mean"), median_best_rank=("best_rank", "median"), n=("hit", "size"),
    inner_hit_rate=("inner_hit", "mean"), median_inner_rank=("inner_best", "median"),
).unstack("lens").round(3)

In [ ]:
fig, axes = plt.subplots(1, len(ROLE_NAMES), figsize=(15, 4), sharey=True, squeeze=False)
for ax, dataset in zip(axes.flat, ROLE_NAMES):
    group = roles[roles.dataset == dataset]
    for (role, name), role_group in group.groupby(["role_name", "lens"]):
        ax.plot(
            (np.stack(role_group.ranks) <= K).mean(0), marker="o", ms=3,
            label=f"{role}: {name}", linestyle="--" if name == "logit lens" else "-",
        )
    ax.set_title(dataset)
    ax.set_xlabel("layer (last = model output)")
    ax.grid(alpha=0.3)
    ax.legend(fontsize=7)
axes.flat[0].set_ylabel(f"rank ≤ {K}")
plt.tight_layout()
plt.show()

### 6d. Intermediate hits by supported next-token target match

In [ ]:
correctness = hits.merge(
    with_target[["dataset", "item", "model_correct"]], on=["dataset", "item"], validate="many_to_one"
)
correctness.groupby(["dataset", "model_correct", "lens"]).hit.agg(["mean", "size"]).unstack(["model_correct", "lens"]).round(3)

## 7. Items: high- and low-ranked supported words

In [ ]:
columns = ["dataset", "item", "lens", "word", "best_rank", "best_layer", "inner_best", "inner_best_layer", "in_prompt"]
inter.sort_values("best_rank").groupby(["dataset", "lens"]).head(3)[columns]

In [ ]:
inter.sort_values("best_rank", ascending=False).groupby(["dataset", "lens"]).head(3)[columns]

What the lens reads at the readout position, layer by layer (first items of each dataset):

In [ ]:
readout = items.groupby(["dataset", "lens"]).head(2).set_index(["dataset", "item", "lens"]).readout_top1.apply(pd.Series)
readout.columns.name = "layer"
readout.sort_index()[shown_layers(model.n_layers, LAYER_STRIDE)]

The most common readout top-1 tokens per layer over all items — generic tokens mean the lens shows no item-specific content there:

In [ ]:
top1_tables = {}
for name in LENS_NAMES:
    top1_by_layer = pd.DataFrame(items[items.lens == name].readout_top1.tolist())
    top1_tables[name] = pd.DataFrame({
        "distinct top-1 / items": top1_by_layer.nunique() / len(top1_by_layer),
        "most common": {
            layer: ", ".join(f"{token!r}×{count}" for token, count in column.value_counts().head(5).items())
            for layer, column in top1_by_layer.items()
        },
    }).rename_axis("layer")
pd.concat(top1_tables, axis=1, names=["lens", "metric"])

### 7a. Paired inner-layer ranks: which lens improves each intermediate?

One row per intermediate occurrence, aligned by dataset, item, word, and role. A positive
`log10 gain` means J-lens lowers the best inner-layer rank; ties are counted explicitly.

In [ ]:
head_to_head = inter.pivot(
    index=["dataset", "item", "word", "role"], columns="lens", values="inner_best"
)
head_to_head["winner"] = np.select(
    [head_to_head["J-lens"] < head_to_head["logit lens"],
     head_to_head["J-lens"] > head_to_head["logit lens"]],
    ["J-lens", "logit lens"], default="tie",
)
head_to_head["log10 gain"] = np.log10(head_to_head["logit lens"]) - np.log10(head_to_head["J-lens"])
paired_summary = head_to_head.groupby("dataset").winner.value_counts().unstack(fill_value=0)
paired_summary = paired_summary.reindex(columns=["J-lens", "logit lens", "tie"], fill_value=0)
paired_summary["median log10 gain"] = head_to_head.groupby("dataset")["log10 gain"].median()
display(paired_summary.round(3))
display(head_to_head.sort_values("log10 gain", ascending=False).head(15))
display(head_to_head.sort_values("log10 gain").head(15))

## Interactive Qwen glossary (optional extra forward)

Exactly as in `walkthrough_orig`: load `assets/qwen_gloss.json.gz`, convert keys to integer
IDs, pass it as `build_page(..., alt_token=gloss)`, and use `compute_slice(..., mask_display=True)`.
Original tokenizer-decoded spellings remain in the page; glosses annotate them. Missing entries
keep their original text. The glossary does not replace the tokenizer or alter scoring.
`mask_display` selects word-like display candidates but ranks stay against the full vocabulary.
There is no tokenizer-revision metadata in this glossary: compatibility is not established.

In [ ]:
import gzip
from jlens.vis import build_page, compute_slice, notebook_iframe

RUN_INTERACTIVE_SLICE = False  # Opt in to one additional model forward.
gloss = None
if ENABLE_QWEN_GLOSS:
    with gzip.open(Path(REPO_DIR) / "assets" / "qwen_gloss.json.gz", "rt", encoding="utf-8") as handle:
        gloss = {int(key): value for key, value in json.load(handle).items()}
    print(f"Loaded {len(gloss):,} display-only glosses; tokenizer provenance unverified.")
if RUN_INTERACTIVE_SLICE:
    slice_prompt = evals["multihop"][0]["prompt"]  # Preserve prompt whitespace.
    slice_data = compute_slice(
        model, fitted_lens, slice_prompt, layer_stride=LAYER_STRIDE,
        mask_display=True,
    )
    page, _, _ = build_page(
        slice_data, slice_prompt, title="Pretrained Qwen J-lens",
        description="Display-only glossary; full-vocabulary ranks.", alt_token=gloss,
    )
    display(notebook_iframe(page))

## 8. Adaptations of reference Figures 52, 55, and 56

These reproduce **plot forms, not the reference experiment**. Figure 52 uses supported
whole-token words only, with coverage reported above. Figures 55/56 use explicitly supplied
independent held-out text and are disabled by default; no fitting or corpus download occurs.

In [ ]:
from jlens.reference_plots import (
    intermediate_rank_sweep, plot_intermediate_sweep,
    plot_distribution_summary, plot_lens_comparison,
)

reference52 = intermediate_rank_sweep(words)
display(reference52.counts)
fig52, axes52 = plot_intermediate_sweep(reference52)
plt.show()

### 8b. Figures 55/56: explicitly supplied held-out text (opt-in extra evaluation)
**No refit is needed.** These distribution metrics cannot be recovered from `words`/`items`;
they require new forwards with the already loaded model/lens. Default is **disabled**. Supply
independent texts below (not the fitting corpus or task prompts) and name their source.
No downloads or automatic corpus substitution occur. Small samples are diagnostics, not
reproductions of pretraining-distribution results. Record selection and fitting provenance.

All actual model blocks are evaluated. Depth = 100 * block_index / (n_blocks - 1): first
block output 0, shared final output 100; no embedding row. Means weight every non-special
input position equally, including each text's last position. Overlength texts raise;
shorten them explicitly to HELD_OUT_MAX_SEQ_LEN tokens rather than silently truncating.
Longer texts contribute more tokens. Natural-log KL and entropy are in **nats**.

* **55:** KL(model || lens), lens entropy, top-1 agreement with model **in percent**.
  J-lens blue, logit lens purple.
* **56:** one purple J/logit comparison at each **same layer**: vocabulary-vector cosine,
  **0.5 * (KL(J || logit) + KL(logit || J))**, and top-1 agreement **as a fraction**.
  The half-sum is explicit; original normalization unknown. Cross-layer pairs are never averaged.
* Cosine compares corresponding rows of W_U and W_U @ J, averaged equally over nonzero
  vocabulary-vector pairs. This **linear-head-only geometry** excludes bias, final normalization,
  and softcap; it is not exact global geometry for a nonlinear decoder. Unsupported output
  heads/adapters are labeled **unavailable, not zero** in the status/counts table. Distributions
  still use `model.unembed`, including its normalization, output head, and any softcap.
  The cosine axis is **0 to 1** for nonnegative/unavailable values; any displayed negative value
  extends it to **-1 to 1**, disclosed in the title. Negative cosines are never silently clipped.

Start with a few short texts: all-layer readouts may be expensive for large models/vocabularies.
Position chunks bound vocabulary-distribution memory, not activation/Jacobian memory. Reduce
chunk size or sequence length if needed. Evaluation and plotting are separate so cached tables
can be replotted without extra forwards.

In [ ]:
# Opt in here; keep this sample independent of lens fitting.
HELD_OUT_TEXTS = None  # a nonempty list[str] of held-out passages
HELD_OUT_SOURCE = ''  # corpus/version, split, sample selection
# Local-file example: one independently held-out passage per nonempty line.
# HELD_OUT_TEXTS = [s for s in Path('/path/to/heldout.txt').read_text().splitlines() if s.strip()]
# HELD_OUT_SOURCE = 'my-corpus v1, held-out split, first 8 passages (disjoint from fitting)'
HELD_OUT_MAX_SEQ_LEN = 128
HELD_OUT_POSITION_CHUNK_SIZE = 4
GEOMETRY_VOCAB_CHUNK_SIZE = 1024

In [ ]:
# Clear cached results first so disabled/failed reruns cannot plot stale metrics.
held_out_metrics = held_out_geometry = held_out_run = None
if HELD_OUT_TEXTS is None:
    print('Figures 55/56 skipped: supply independent HELD_OUT_TEXTS and HELD_OUT_SOURCE above. No downloads or extra evaluation run.')
else:
    if not isinstance(HELD_OUT_TEXTS, (list, tuple)) or not HELD_OUT_TEXTS:
        raise ValueError('Supply a nonempty list/tuple of independently held-out strings.')
    if not all(isinstance(text, str) and text.strip() for text in HELD_OUT_TEXTS):
        raise ValueError('Every held-out passage must be a nonempty string.')
    if not isinstance(HELD_OUT_SOURCE, str) or not HELD_OUT_SOURCE.strip():
        raise ValueError('Describe the independent sample in HELD_OUT_SOURCE first.')
    from jlens.metrics import evaluate_distributions, lens_vector_geometry

    held_out_run = {
        'model': globals().get('MODEL_ID', type(model).__name__),
        'model_dtype': str(next(model.layers[0].parameters()).dtype),
        'device': str(model.input_device),
        'source': HELD_OUT_SOURCE, 'n_texts_supplied': len(HELD_OUT_TEXTS),
        'max_seq_len': HELD_OUT_MAX_SEQ_LEN,
        'position_chunk_size': HELD_OUT_POSITION_CHUNK_SIZE,
        'layers': 'all blocks, shared final output', 'weighting': 'token',
        'float32_matmul_precision': torch.get_float32_matmul_precision(),
    }
    display(held_out_run)
    held_out_metrics = evaluate_distributions(
        model, fitted_lens, HELD_OUT_TEXTS, layers=None,
        max_seq_len=HELD_OUT_MAX_SEQ_LEN,
        position_chunk_size=HELD_OUT_POSITION_CHUNK_SIZE, pairwise=True,
    )
    held_out_geometry = lens_vector_geometry(
        model, fitted_lens, layers=None, vocab_chunk_size=GEOMETRY_VOCAB_CHUNK_SIZE,
    )
    display(held_out_metrics.layers)
    display(held_out_geometry)

In [ ]:
fig55 = axes55 = fig56 = axes56 = None
if held_out_metrics is None or held_out_geometry is None:
    print('No held-out figures to plot. Configure the sample and run the evaluation cell first.')
else:
    print('Held-out sample:', held_out_run['source'])
    fig55, axes55 = plot_distribution_summary(held_out_metrics.layers, n_layers=model.n_layers)
    plt.show()
    fig56, axes56 = plot_lens_comparison(
        held_out_metrics.pairs, held_out_geometry, n_layers=model.n_layers,
    )
    plt.show()

## 9. Conclusions after running

Interpret supported-only rank improvements alongside coverage and control rates. These
readouts are probes, not evidence that the model computed a particular reasoning step.

In [ ]:
def pct(x):
    return "n/a" if x is None or pd.isna(x) else f"{100 * x:.0f}%"


def layer(x):
    return "never" if x is None else f"L{x}"


def hit_rates(frame, column):
    """Hit rate (best rank <= K) split by a boolean column: {value: (rate, n)}."""
    hit = (frame.best_rank <= K).groupby(frame[column])
    return {value: (rate, n) for value, rate, n in zip(hit.mean().index, hit.mean(), hit.size())}

In [ ]:
print("Supported next-token target match (not generated-answer accuracy):")
display(coverage)
print(f"Paired inner-layer rank comparison at K={K}:")
display(paired_summary)
print("External lens fitting provenance remains UNVERIFIED.")
print("Compare inner-layer intermediates against controls; report coverage with every score.")

### Limitations

* Complete decoded-token variants are a lexical probe, not a generated-answer benchmark.
  Multi-token targets without an accepted complete token remain unscored. Context-sensitive
  tokenization and the likelihood of full continuations are not evaluated.
* Both lenses share final model logits; a pretrained lens cannot improve final model accuracy.
  All-layer scores include that shared output. Inspect inner-layer and control comparisons.
* Shape/finite checks and identity/logit checks do not validate external fitting provenance.
  Training revision, numerical precision, BOS policy, and corpus compatibility remain unknown.
* The Qwen glossary is a best-effort, unverified token-ID mapping. It changes display only;
  word-like masking affects displayed candidates, not their full-vocabulary ranks or metrics.
* Replacement models may have low/zero spelling coverage; the tables remain authoritative,
  and some plotting strata may then be empty. Scores are not comparable to legacy prefix scores.
* No inference was run while creating this notebook. Outputs must be generated on a suitable
  runtime. Save the executed notebook and `run_metadata` when reporting experimental results.